# Data preparation and feature engineering
This notebook is split from the original thesis workflow to keep each stage focused and easier to maintain.


In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
from pathlib import Path
import os
_here = Path.cwd().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)
elif not (_here / "image").is_dir() and (_here.parent / "image").is_dir():
    os.chdir(_here.parent)
print("Project root:", Path.cwd())


In [ ]:
import pandas as pd
import os

# List of match folder names
match_folders = ['match1', 'match2', 'match3', 'match4', 'match5','match6','match7','match8','match9','match10','match11']

# Initialize empty lists to store DataFrames from each folder
coordinates_list = []
passes_list = []

# Loop through each match folder and read the corresponding CSV files
for folder in match_folders:
    coordinates_path = os.path.join(folder, 'coordinates.csv')
    passes_path = os.path.join(folder, 'passes.csv')
    
    if os.path.exists(coordinates_path):
        df_coord = pd.read_csv(coordinates_path)
        df_coord['match'] = folder  # Add match identifier column
        coordinates_list.append(df_coord)

    if os.path.exists(passes_path):
        df_pass = pd.read_csv(passes_path)
        df_pass['match'] = folder  # Add match identifier column
        passes_list.append(df_pass)

# Concatenate all match DataFrames into a single DataFrame
all_coordinates = pd.concat(coordinates_list, ignore_index=True)
all_passes = pd.concat(passes_list, ignore_index=True)

# Write the merged DataFrames into an Excel file with two sheets
output_path = 'merged_matches_plus.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    all_coordinates.to_excel(writer, sheet_name='coordinates', index=False)
    all_passes.to_excel(writer, sheet_name='passes', index=False)

print(f'Saved as {output_path}')


In [ ]:
import pandas as pd
df_passes = pd.read_excel('merged_matches_plus.xlsx', sheet_name='passes')
df_passes['timestamp'] = pd.to_datetime(df_passes['timestamp'], unit='ms')
df_coordinates = pd.read_excel('merged_matches_plus.xlsx', sheet_name='coordinates', parse_dates=['timestamp'])


# Calculate Speed

In [ ]:
import numpy as np

# Ensure the coordinates table is sorted by timestamp
df_coordinates = df_coordinates.sort_values('timestamp')

# Define a function to compute pass speed from coordinates
def compute_pass_speed_from_coordinates(pass_time):
    # Find the index of the current pass timestamp in the coordinates table
    i = df_coordinates['timestamp'].searchsorted(pass_time)
    
    # If the index is out of range, return NaN
    if i >= len(df_coordinates) - 1:
        return np.nan

    x1, y1 = df_coordinates.iloc[i]['ballposX'], df_coordinates.iloc[i]['ballposY']
    t1 = df_coordinates.iloc[i]['timestamp']
    
    x2, y2 = df_coordinates.iloc[i+4]['ballposX'], df_coordinates.iloc[i+1]['ballposY']  # mixed step
    t2 = df_coordinates.iloc[i+4]['timestamp']
    
    # Time difference
    dt = (t2 - t1).total_seconds()
    if dt == 0:
        return np.nan
    
    # Euclidean distance
    dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    return dist / dt

# Apply the speed calculation to each pass event
df_passes['pass_speed'] = df_passes['timestamp'].apply(compute_pass_speed_from_coordinates)


In [ ]:
# Remove ball speeds = 0 or > 40
df_passes_cleaned = df_passes[
    (df_passes['pass_speed'] > 0) & (df_passes['pass_speed'] <= 40)
].copy()

In [ ]:
df_passes_cleaned = df_passes_cleaned[df_passes_cleaned['passedPlayer_Zone'] != '0.0']
# Construct the mapping dictionary
area_map = {'Defence': 0, 'Mid field': 1, 'Attack': 2}
pass_type_map = {'Backward pass': 0, 'Lateral pass': 1, 'Forward pass': 2}
pressure_dir_map = {'back': 0, 'front': 1}
pressure_level_map = {
    'No Pressure': 0,
    'Limited Pressure': 1,
    'Full Pressure': 2
}

df_passes_cleaned['pass_area_num'] = df_passes_cleaned['passedPlayer_Zone'].map(area_map)
df_passes_cleaned['pass_direction_num'] = df_passes_cleaned['receivedPlayerId_PassType'].map(pass_type_map)
df_passes_cleaned['pressure_direction_num'] = df_passes_cleaned['Pressure direction'].map(pressure_dir_map)
df_passes_cleaned['pressure_level_num'] = df_passes_cleaned['Pressure level'].map(pressure_level_map)


# Construct Window_Score

In [ ]:



import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# NOTE: comment translated/cleaned for English-only repository.
df = pd.read_csv("label_annotation_sample (1).csv")

# NOTE: comment translated/cleaned for English-only repository.
features = [ 'pass_speed', 'receivedPlayerId_PassLength',
    'pressure_level_num', 'pressure_direction_num',
    'pass_area_num', 'pass_direction_num',
    'xT_change']
X = df[features]
y = df['human_label']

# NOTE: comment translated/cleaned for English-only repository.
model = LogisticRegression()
model.fit(X, y)

# NOTE: comment translated/cleaned for English-only repository.
weights = pd.Series(model.coef_[0], index=features)
print("Logistic               :")
print(weights)
weights.sort_values().plot(kind='barh', color='skyblue')
plt.title("Logistic Regression Weights (Predicting Human Label)")
plt.xlabel("Weight (Positive = More Valuable)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df=df_passes_cleaned.copy()

df['xT_change'] = df['receivedPlayer_xT_gained']


# Drop rows with missing values in key columns
required_cols = ['xT_change', 'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num']
df = df.dropna(subset=required_cols)

# Map pressure level to score
df['pressure_score'] = df['pressure_level_num'].map({
    0: 0.0,
    1: 0.5,
    2: 1.0
})

# Calculate window score based on multiple features
df['window_score'] = (
    0.4 * (df['xT_change'].clip(0, 0.05) / 0.05) +
    0.25 * (df['pass_speed'].clip(0, 10) / 10) +
    0.2 * (df['receivedPlayerId_PassLength'].clip(0, 15) / 15) +
    0.15 * df['pressure_score']
)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
sns.histplot(df['window_score'], bins=30, kde=True)
plt.title('Distribution of Window Score')
plt.xlabel('window_score')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)
print(df['is_valuable'].value_counts())